# Agents with smolagents

<sup>This notebook is a part of the Natural Language Processing class at the University of Ljubljana, Faculty of Computer and Information Science.</sup>

---

So far in this course we have seen how to **prompt** an LLM (single turn) and how to **augment** it with retrieved context (RAG). Both approaches share a common limitation: the model receives an input, runs one forward pass, and produces a single output. It cannot loop, backtrack, or use tools dynamically.

**Agents** remove this limitation. An agent is a system in which the LLM acts as a reasoning engine that decides, step by step, what to do next - querying tools, inspecting results, and repeating until the task is complete. This is the architecture behind modern AI assistants that browse the web, write and execute code, and orchestrate multi-step workflows.

We use **[smolagents](https://github.com/huggingface/smolagents)** - a lightweight HuggingFace library that provides two agent types, a rich set of built-in tools, and clean primitives for building custom tools and multi-agent pipelines. Here we drive the agents with OpenAI's **`gpt-4o-mini`** through the API, so there is no local GPU or model download - you only need your `OPENAI_API_KEY`.

We cover:
1. What is an agent? (observe -> think -> act)
2. Setup: smolagents, imports, and connecting the OpenAI model
3. Your first agent - a bare `CodeAgent` with no tools
4. Built-in tools (`PythonInterpreterTool`, `FinalAnswerTool`)
5. Writing a custom tool with the `@tool` decorator
6. Tool composition - chaining multiple tools in one task
7. `CodeAgent` vs `ToolCallingAgent` - comparing internal reasoning
8. Inspecting the agent's memory
9. Multi-step reasoning - tasks requiring sequential steps
10. Controlling the agent - `max_steps` and graceful failure
11. Advanced: multi-agent systems with `managed_agents`

## 1  What is an Agent?

A **plain LLM call** maps one prompt to one response. An **agent** wraps that LLM in a loop:

```
User query
    │
    ▼
┌───────────────────────────────┐
│  OBSERVE — read task / tool   │
│           result              │
│                               │
│  THINK   — LLM reasons about  │
│           what to do next     │
│                               │
│  ACT     — call a tool, write │
│           code, or answer     │
└───────────┬───────────────────┘
            │ repeat until done
            ▼
      Final answer
```

### The three parts of an agentic system

| Component | Role |
|---|---|
| **LLM (brain)** | Reads context, reasons, decides the next action |
| **Tools** | External functions the LLM can call (calculator, search, code runner, …) |
| **Executor / loop** | Runs the tool, feeds results back to the LLM, detects when the task is done |

### Agent vs plain LLM call

| | Plain LLM | Agent |
|---|---|---|
| Steps | 1 (prompt → response) | N (loop until done) |
| Memory | None (stateless) | Step log keeps all prior observations |
| Tools | None | Any callable function |
| Self-correction | Impossible | Can retry after an error |

The ReAct paper (Yao et al., 2022) showed that interleaving **reasoning** ("Thought") with **actions** (tool calls) dramatically improves performance on tasks that require planning or factual look-ups. Modern agent frameworks — including smolagents — implement exactly this pattern under the hood.

## 2  Setup

We only need a few lightweight libraries - all model inference runs through the **OpenAI API**, so there is no `torch`, `transformers`, or GPU involved.

| Library | Purpose |
|---|---|
| `smolagents` | Agent framework: `CodeAgent`, `ToolCallingAgent`, built-in tools, `@tool` decorator |
| `openai` | Client used by smolagents' `OpenAIServerModel` to call the OpenAI API |
| `duckduckgo-search` | Backend for the optional `DuckDuckGoSearchTool` web-search demo |
| `python-dotenv` | Loads your `OPENAI_API_KEY` from a local `.env` file |

In [ ]:
# !pip install -q -U smolagents openai ddgs python-dotenv

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

# Load OPENAI_API_KEY from a local .env file (which must NOT be committed).
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found - add it to your .env file."

from smolagents import (
    CodeAgent,
    ToolCallingAgent,
    OpenAIServerModel,
    PythonInterpreterTool,
    FinalAnswerTool,
    DuckDuckGoSearchTool,
    tool,
)
from smolagents.memory import ActionStep, FinalAnswerStep, SystemPromptStep, TaskStep

## 3  Connecting the Model

We drive every agent in this notebook with OpenAI's **`gpt-4o-mini`**, accessed through smolagents' `OpenAIServerModel`. The model runs on OpenAI's servers, so there is no download, GPU, or quantisation step - only your `OPENAI_API_KEY` (loaded above from `.env`).

In [3]:
MODEL_ID = "gpt-4o-mini"

model = OpenAIServerModel(
    model_id=MODEL_ID,
    api_key=os.environ["OPENAI_API_KEY"],
)

print(f"Model ready: {MODEL_ID}")

Model ready: gpt-4o-mini


`OpenAIServerModel` wraps the OpenAI Chat Completions API in the `smolagents` `Model` interface, handling chat formatting and tool-call parsing automatically. We create it **once here** and pass it to every agent we build below. To use a different model (e.g. `gpt-4o`), just change `MODEL_ID`.

## 4  Your First Agent

The simplest possible agent: `CodeAgent(tools=[], model=model)`. `tools=[]` does **not** mean the agent is powerless — it means there are no pre-wrapped smolagents tools. The agent still has full access to the **Python executor**: it can import any installed package, call external APIs, manipulate files, and run arbitrary computations. The executor *is* the capability.

`CodeAgent` works by asking the LLM to write **Python code** that solves each step. The executor runs that code and feeds the output back to the LLM as the next observation — making the entire Python ecosystem available as a free-form tool.

In [4]:
agent = CodeAgent(
    tools=[],
    model=model,
    code_block_tags="markdown",
    executor_type="local",
    max_steps=5,
    stream_outputs=True
)

print(agent.system_prompt)

result = agent.run(
    "If a snail travels 3.7 cm per minute, how many minutes does it need "
    "to cross a 2.5 m garden path? Round to the nearest whole minute."
)
print("\nFinal answer:", result)


You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.

At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '```python', and closed with '```'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.

Here are a few examples u

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ If a snail travels 3.7 cm per minute, how many minutes does it need to cross a 2.5 m garden path? Round to the  │
│ nearest whole minute.                                                                                           │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Step 1: Convert meters to centimeters                                                                          
  distance_m = 2.5  # garden path length in meters                                                                 
  distance_cm = distance_m * 100  # convert to centimeters                                                         
                                                                                                                   
  # Step 2: Calculate the time in minutes                                                                          
  speed_cm_per_min = 3.7  # speed of the snail in centimeters per minute                                           
  time_minutes = distance_cm / speed_cm_per_min  # time in minutes                                                 
                                                                                                                   
  # Step 3: Round to the nearest whole minute                                                                      
  time_minutes_rounded = round(time_minutes)                                                                       
  print(time_minutes_rounded)                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
68

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(68)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 68

[Step 2: Duration 1.40 seconds| Input tokens: 4,399 | Output tokens: 279]


Final answer: 68


The agent's reasoning trace appears in the output above. Notice the structure:

1. **System prompt step** — smolagents injects the tool list and instructions.
2. **Task step** — your question is added to the conversation.
3. **Action step(s)** — the LLM emits a code block; the executor runs it; the result becomes the observation.
4. **Final answer step** — the LLM calls `final_answer(...)` to signal it is done.

Even with `tools=[]` the agent wrote valid Python to perform the unit conversion and returned the correct answer.

> **How `CodeAgent` works under the hood**
> 
> At each step the LLM is asked to produce a Python code block. The local executor runs that block in a sandboxed namespace, captures `print` output and the return value, and returns them as the next observation. The LLM can therefore use any Python expression — arithmetic, string formatting, data structures — as a free-form calculator without relying on dedicated tools.

## 5  Built-in Tools

smolagents ships with a set of ready-made tools. The most commonly used ones:

| Tool class | What it does |
|---|---|
| `PythonInterpreterTool` | Runs arbitrary Python code (explicit tool-call flavour) |
| `FinalAnswerTool` | Signals the end of the run and sets the return value |
| `DuckDuckGoSearchTool` | Web search via DuckDuckGo (requires internet access) |
| `WikipediaSearchTool` | Searches Wikipedia articles |
| `VisitWebpageTool` | Fetches and parses a URL |

Tools are passed as a **list** to the agent constructor. The model reads each tool's name and docstring to decide when to call it — so **tool descriptions matter** (see callout below).

Let's demonstrate `PythonInterpreterTool` and `FinalAnswerTool` on a task that requires real computation:

In [5]:
agent_tools = CodeAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=6,
    additional_authorized_imports=["numpy"],   # Agent may import numpy in its generated code
)

result = agent_tools.run(
    "Compute the sum of all prime numbers less than 50. Use numpy module."
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Compute the sum of all prime numbers less than 50. Use numpy module.                                            │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import numpy as np                                                                                               
                                                                                                                   
  # Function to check if a number is prime                                                                         
  def is_prime(num):                                                                                               
      if num < 2:                                                                                                  
          return False                                                                                             
      for i in range(2, int(np.sqrt(num)) + 1):                                                                    
          if num % i == 0:                                                                                         
              return False                                                                                         
      return True                                                                                                  
                                                                                                                   
  # Generate an array of numbers from 2 to 49                                                                      
  numbers = np.arange(2, 50)                                                                                       
                                                                                                                   
  # Filter the prime numbers                                                                                       
  prime_numbers = np.array(list(filter(is_prime, numbers)))                                                        
                                                                                                                   
  # Calculate the sum of the prime numbers                                                                         
  sum_of_primes = np.sum(prime_numbers)                                                                            
  print(sum_of_primes)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
328

Out: None

[Step 1: Duration 4.79 seconds| Input tokens: 2,089 | Output tokens: 224]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(328)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 328

[Step 2: Duration 1.44 seconds| Input tokens: 4,603 | Output tokens: 273]


Final answer: 328


With `PythonInterpreterTool` available the agent can write and execute code snippets as explicit **tool calls** rather than inline scratchpad code. `FinalAnswerTool` gives it a clean way to commit to a final value.

> **Why tool descriptions matter**
>
> The LLM never sees the tool's source code — only its **name** and **docstring**. These two pieces of text are what the model uses to decide whether to call a tool and how to pass arguments. A vague docstring like `"Does something with text"` will cause the model to ignore the tool or misuse it. A precise description like `"Counts words, sentences, and characters in a text string"` makes the tool reliably discoverable.

## 6  Writing a Custom Tool

Any Python function decorated with `@tool` becomes a first-class smolagents tool. The decorator introspects the function's **name**, **docstring**, and **type annotations** to build the tool description that the model reads.

Rules for a well-formed tool:
- All parameters must have type annotations.
- The docstring must describe what the tool does *and* document each argument (NumPy or Google style).
- The return type should be one of `str`, `int`, `float`, `bool`, or `dict`.

In [6]:
@tool
def text_statistics(text: str) -> dict:
    """Compute basic statistics for a piece of text.

    Returns a dictionary with:
    - word_count (int): number of words
    - sentence_count (int): number of sentences
    - avg_word_length (float): average word length
    - char_count (int): total character count excluding whitespace

    Args:
        text: The input text to analyse.
    """
    words = text.split()
    sentences = max(1, text.count(".") + text.count("!") + text.count("?"))
    avg_word_len = sum(len(w.strip(".,!?")) for w in words) / max(1, len(words))
    return {
        "word_count": len(words),
        "sentence_count": sentences,
        "avg_word_length": round(avg_word_len, 2),
        "char_count": sum(len(w) for w in words),
    }


# Inspect what the model will see
print("Tool name  :", text_statistics.name)
print("Description:", text_statistics.description)
print("Inputs     :", text_statistics.inputs)
print("Output type:", text_statistics.output_type)

Tool name  : text_statistics
Description: Compute basic statistics for a piece of text.

Returns a dictionary with:
- word_count (int): number of words
- sentence_count (int): number of sentences
- avg_word_length (float): average word length
- char_count (int): total character count excluding whitespace
Inputs     : {'text': {'type': 'string', 'description': 'The input text to analyse.'}}
Output type: object


In [7]:
agent_custom = CodeAgent(
    tools=[text_statistics, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=5,
)

passage = (
    "The transformer architecture was introduced in 2017. "
    "It relies on a self-attention mechanism that computes relationships "
    "between all tokens in a sequence simultaneously. "
    "This parallelism made training much faster than previous RNN approaches."
)

result = agent_custom.run(
    f"Use the text_statistics tool to analyse the following passage, then "
    f"report the word count and the average word length:\n\n{passage}"
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Use the text_statistics tool to analyse the following passage, then report the word count and the average word  │
│ length:                                                                                                         │
│                                                                                                                 │
│ The transformer architecture was introduced in 2017. It relies on a self-attention mechanism that computes      │
│ relationships between all tokens in a sequence simultaneously. This parallelism made training much faster than  │
│ previous RNN approaches.                                                                                        │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  passage = "The transformer architecture was introduced in 2017. It relies on a self-attention mechanism that     
  computes relationships between all tokens in a sequence simultaneously. This parallelism made training much      
  faster than previous RNN approaches."                                                                            
  stats = text_statistics(passage)                                                                                 
  word_count = stats['word_count']                                                                                 
  avg_word_length = stats['avg_word_length']                                                                       
  print(f"Word Count: {word_count}, Average Word Length: {avg_word_length}")                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Word Count: 33, Average Word Length: 6.27

Out: None

[Step 1: Duration 3.30 seconds| Input tokens: 2,110 | Output tokens: 139]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({"word_count": 33, "average_word_length": 6.27})                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'word_count': 33, 'average_word_length': 6.27}

[Step 2: Duration 2.09 seconds| Input tokens: 4,527 | Output tokens: 213]


Final answer: {'word_count': 33, 'average_word_length': 6.27}


The `@tool` decorator requires no boilerplate class definition — a plain annotated function is sufficient. The agent correctly identified `text_statistics` as the right tool, called it with the passage as argument, and summarised the returned dictionary.

## 7  Tool Composition

The real power of agents emerges when they must **chain multiple tools** in sequence. The agent decides on its own which tool to call at each step, in what order, and how to pass intermediate results forward.

We give the agent three tools and pose a task that requires at least two of them in a specific order:

In [8]:
@tool
def word_frequency(text: str, top_n: int) -> dict:
    """Return the top N most frequent words in a text, ignoring case.

    Args:
        text: The input text to analyse.
        top_n: How many top words to return.
    """
    from collections import Counter
    tokens = [w.strip(".,!?;:\"\'()").lower() for w in text.split()]
    tokens = [t for t in tokens if len(t) > 2]   # Drop very short words
    return dict(Counter(tokens).most_common(top_n))


agent_composed = CodeAgent(
    tools=[text_statistics, word_frequency, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=8,
)

corpus = (
    "Natural language processing enables computers to understand human language. "
    "Language models learn statistical patterns from large language corpora. "
    "Understanding language structure is fundamental to natural language processing."
)

result = agent_composed.run(
    "First compute the text statistics for the corpus below, then find the top 3 "
    "most frequent words. Report both results.\n\n" + corpus
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ First compute the text statistics for the corpus below, then find the top 3 most frequent words. Report both    │
│ results.                                                                                                        │
│                                                                                                                 │
│ Natural language processing enables computers to understand human language. Language models learn statistical   │
│ patterns from large language corpora. Understanding language structure is fundamental to natural language       │
│ processing.                                                                                                     │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  corpus = """Natural language processing enables computers to understand human language. Language models learn    
  statistical patterns from large language corpora. Understanding language structure is fundamental to natural     
  language processing."""                                                                                          
                                                                                                                   
  text_stats = text_statistics(corpus)                                                                             
  print("Text Statistics:", text_stats)                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Text Statistics: {'word_count': 27, 'sentence_count': 3, 'avg_word_length': 7.33, 'char_count': 201}

Out: None

[Step 1: Duration 3.88 seconds| Input tokens: 2,157 | Output tokens: 138]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({                                                                                                   
      "text_statistics": {'word_count': 27, 'sentence_count': 3, 'avg_word_length': 7.33, 'char_count': 201},      
      "top_words": {'language': 6, 'natural': 2, 'processing': 2}                                                  
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'text_statistics': {'word_count': 27, 'sentence_count': 3, 'avg_word_length': 7.33, 'char_count': 
201}, 'top_words': {'language': 6, 'natural': 2, 'processing': 2}}

[Step 3: Duration 2.38 seconds| Input tokens: 7,217 | Output tokens: 331]


Final answer: {'text_statistics': {'word_count': 27, 'sentence_count': 3, 'avg_word_length': 7.33, 'char_count': 201}, 'top_words': {'language': 6, 'natural': 2, 'processing': 2}}


The tool-call log shows the agent's execution order. It called `text_statistics` first, read the output, then called `word_frequency` — exactly the chain required by the task. The agent inferred this ordering from the task description alone, without any explicit sequencing instructions.

## 8  Web Search with `DuckDuckGoSearchTool`

One of the most useful built-in tools is `DuckDuckGoSearchTool`, which lets the agent query the web at runtime — grounding its answers in live information rather than model weights.

| Property | Detail |
|---|---|
| Class | `smolagents.DuckDuckGoSearchTool` |
| Requires | Internet access; no API key needed |
| Returns | A string with the top search results (titles + snippets) |
| Best for | Factual look-ups, current events, anything that may postdate the model |

You can call the tool directly to inspect its output before involving an agent:


In [9]:
search = DuckDuckGoSearchTool()

# Call the tool directly — useful for debugging what the agent will see
results = search("smolagents HuggingFace library")
print(results[:800])


## Search Results

[GitHub - huggingface/smolagents: 🤗 smolagents: a barebones](https://github.com/huggingface/smolagents)
Misc { smolagents , title = { `smolagents`: a smol library to build great agentic systems. ... url{https://github.com/huggingface/smolagents ...

[Smolagents : Huggingface AI Agent Framework](https://smolagents.org/)
I spent my weekend learning about agentic workflows and playing around with the smolagents library released by @huggingface .

[huggingface/smolagents | DeepWiki](https://deepwiki.com/huggingface/smolagents)
smolagents is a Python library for building LLM-powered agents. ... The library is published on PyPI as smolagents .

[huggingface/smolagents - Gitstar Ranking](https://gitstar-ranking.com/huggingface/smolagents)
huggingface / smolagents ... smolagents


### 8.1  Agent with web search

Now we hand the tool to a `CodeAgent`. The agent decides *when* to search and *what* to search for — we only specify the goal:


In [10]:
agent_search = CodeAgent(
    tools=[DuckDuckGoSearchTool(), FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=5,
)

result = agent_search.run(
    "What is the latest stable version of the smolagents library? "
    "Search for it and report the version number."
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the latest stable version of the smolagents library? Search for it and report the version number.       │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_result = web_search(query="latest stable version of smolagents library")                                  
  print(search_result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[smolagents: a barebones library for agents that think in code. - 
GitHub](https://github.com/huggingface/smolagents)
@Misc{smolagents, title = {`smolagents`: a smol library to build great agentic systems.}, ... v1.26.0 Latest. on 
May 28 · + 35 releases · Packages 0. Uh oh ...

[smolagents - Hugging Face](https://huggingface.co/docs/smolagents/en/index)
smolagents is an open-source Python library designed to make it extremely easy to build and run agents using just a
few lines of code.

[Smolagents : Huggingface AI Agent Framework](https://smolagents.org/)
May 4, 2020 ... Free Code Genetrator with Deepseek v3 latest version → ... The new smolagents library released 
today by @huggingface looks really impressive.

[Hugging Face Just Released SmolAgents: A Smol Library ... - 
Reddit](https://www.reddit.com/r/machinelearningnews/comments/1hq6itb/hugging_face_just_released_smolagents_a_smol/
)
Dec 31, 2024 ... Hugging Face's SmolAgents takes the complexity out of creating intelligent agents. With this new 
toolkit, developers can build agents with built-in search ...

[smolagents-go command - github.com/epuerta9 ... - Go 
Packages](https://pkg.go.dev/github.com/epuerta9/smolagents-go)
Mar 13, 2025 ... Opens a new window with list of versions in this module. Latest Latest Warning. This package is 
not in the latest version of its module. Go ...

[Install and Run Powerful smolagents library and AI Agents by Using 
...](https://www.youtube.com/watch?v=kbsNnWsXap4)
Jan 7, 2025 ... ... version of a larger Llama LLM model with almost the same performance as larger Llama LLMs. You 
can also use the ideas presented in this ...

[Smolagents vs LangGraph: Which One's Easier to Build and Run AI 
...](https://www.zenml.io/blog/smolagents-vs-langgraph)
Sep 28, 2025 ... Smolagents vs LangGraph: Framework Maturity and Lineage ; First Public Release, Dec 2024, January 
2024 ; GitHub Stars, 23,000+, 19,000+ ; Forks ...

[New Framework smolagents - Beginners - Hugging Face 
Forums](https://discuss.huggingface.co/t/new-framework-smolagents/135421)
Jan 13, 2025 ... This library is the simplest framework out there to build powerful agents! By the way, wtf are 
“agents”? We provide our definition in this ...

[Exploring the smolagents Library: A Deep Dive into MultiStepAgent 
...](https://kargarisaac.medium.com/exploring-the-smolagents-library-a-deep-dive-into-multistepagent-codeagent-and-
toolcallingagent-03482a6ea18c)
Feb 8, 2025 ... This ensures that the agent starts with a clean slate for the new task. if reset: 
self.memory.reset() self.monitor.reset(). Resetting the memory ...

[SmolAgents](https://cobusgreyling.substack.com/p/smolagents)
Feb 11, 2025 ... Thanks for reading Cobus Greyling on LLMs, NLU, NLP, chatbots & voicebots! Subscribe for free to 
receive new posts and support my work.

Out: None

[Step 1: Duration 3.15 seconds| Input tokens: 2,031 | Output tokens: 69]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  latest_version = "1.26.0"                                                                                        
  final_answer(latest_version)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 1.26.0

[Step 2: Duration 2.17 seconds| Input tokens: 4,965 | Output tokens: 156]


Final answer: 1.26.0


The agent issued one or more search queries, read the returned snippets, and extracted the version number from the text — all without any retrieval code written by us. Contrast this with the RAG notebook where we had to build the retriever manually: here the retrieval step is entirely delegated to the tool and the agent's reasoning loop.

> **When to use search vs RAG**
>
> | | `DuckDuckGoSearchTool` | RAG (custom retriever) |
> |---|---|---|
> | Knowledge source | Live web | Your own documents |
> | Latency | Network round-trip | Local (fast) |
> | Control | None — results vary | Full — you index the corpus |
> | Best for | Current events, open-domain facts | Private data, reproducibility |


## 9  `CodeAgent` vs `ToolCallingAgent`

smolagents provides two distinct agent types that differ in *how* the LLM expresses its actions:

| | `CodeAgent` | `ToolCallingAgent` |
|---|---|---|
| Action format | Python code block | JSON tool-call object |
| Execution | Local Python interpreter runs the code | Framework calls the matching tool directly |
| Flexibility | Can use any Python expression | Constrained to declared tool signatures |
| Verbosity | Higher (code + output) | Lower (structured JSON) |
| Best for | Tasks needing complex logic or loops | Tasks with well-defined, discrete tool calls |

> **CodeAgent executes Python; ToolCallingAgent emits JSON**
>
> With `CodeAgent`, the model writes `result = text_statistics(passage)` and the executor runs it as real Python. With `ToolCallingAgent`, the model emits `{"name": "text_statistics", "arguments": {"text": "..."}}` and the framework dispatches the call. Both patterns implement the same Thought → Action → Observation loop — they differ only in the action encoding.

Let's run the same task with both types and compare:

In [11]:
TASK = "What is 123 multiplied by 456? Use the python_interpreter tool to compute it."

# --- CodeAgent ---
code_agent = CodeAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=4
)

result_code = code_agent.run(TASK)
print("CodeAgent answer:", result_code)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 123 multiplied by 456? Use the python_interpreter tool to compute it.                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = 123 * 456                                                                                               
  result                                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: 56088

[Step 1: Duration 3.66 seconds| Input tokens: 2,089 | Output tokens: 53]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(56088)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 56088

[Step 2: Duration 1.44 seconds| Input tokens: 4,294 | Output tokens: 99]

CodeAgent answer: 56088


`ToolCallingAgent` emits structured JSON tool calls. `gpt-4o-mini` supports this natively, so we can reuse the exact same model object.

In [12]:
# Reuse the same OpenAI model; it emits JSON tool calls for the ToolCallingAgent.
model_tool = model

In [13]:
TASK = "What is 123 multiplied by 456? Use the python_interpreter tool to compute it."

# --- ToolCallingAgent ---
tc_agent = ToolCallingAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool()],
    model=model_tool,
    max_steps=4,
)

print(tc_agent.system_prompt)

result_tc = tc_agent.run(TASK)
print("ToolCallingAgent answer:", result_tc)

You are an expert assistant who can solve any task using tool calls. You will be given a task to solve as best you can.
To do so, you have been given access to some tools.

The tool call you write is an action: after the tool is executed, you will get the result of the tool call as an "observation".
This Action/Observation can repeat N times, you should take several steps when needed.

You can use the result of the previous action as input for the next action.
The observation will always be a string: it can represent a file, like "image_1.jpg".
Then you can use it as input for the next action. You can do it for instance as follows:

Observation: "image_1.jpg"

Action:
{
  "name": "image_transformer",
  "arguments": {"image": "image_1.jpg"}
}

To provide the final answer to the task, use an action blob with "name": "final_answer" tool. It is the only way to complete the task, else you will be stuck on a loop. So your final output should look like this:
Action:
{
  "name": "final_answer"

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 123 multiplied by 456? Use the python_interpreter tool to compute it.                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'python_interpreter' with arguments: {'code': '123 * 456'}                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Stdout:

Output: 56088

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '56088'}                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 56088

Final answer: 56088

[Step 2: Duration 0.86 seconds| Input tokens: 2,249 | Output tokens: 33]

ToolCallingAgent answer: 56088


In [14]:
# Compare the raw model output for the first action step
def first_action_output(agent):
    for step in agent.memory.steps:
        if isinstance(step, ActionStep) and step.model_output:
            return step.model_output.strip()
    return "(no action step found)"

print("=== CodeAgent — model output (first action) ===")
print(first_action_output(code_agent))

print("\n=== ToolCallingAgent — model output (first action) ===")
print(first_action_output(tc_agent))

=== CodeAgent — model output (first action) ===
Thought: I will use the `python_interpreter` tool to compute the multiplication of 123 and 456. I need to format the code correctly to perform this simple calculation.

```python
result = 123 * 456
result
```

=== ToolCallingAgent — model output (first action) ===
(no action step found)


The `CodeAgent` output contains a Python code block (`` ```py ... ``` ``). The `ToolCallingAgent` output contains a structured tool-call (JSON or a tagged block). Both arrive at the same final answer via different internal representations.

## 10  Inspecting the Agent's Memory

After a run, `agent.memory.steps` holds the full execution history as a list of typed step objects:

| Step type | When it appears |
|---|---|
| `SystemPromptStep` | Always first — contains the injected system prompt |
| `TaskStep` | The user's task |
| `ActionStep` | Each LLM reasoning + tool-execution cycle |
| `PlanningStep` | When `planning_interval` is set (optional planning phase) |
| `FinalAnswerStep` | The terminal step with the final output |

In [15]:
# Re-use code_agent from the previous section
print(f"Total steps in memory: {len(code_agent.memory.steps)}\n")

for i, step in enumerate(code_agent.memory.steps):
    step_type = type(step).__name__
    print(f"[{i}] {step_type}")

    if isinstance(step, SystemPromptStep):
        # Show just the first line of the system prompt
        print("    ", step.system_prompt.splitlines()[0][:80])

    elif isinstance(step, TaskStep):
        print("    Task:", step.task[:80])

    elif isinstance(step, ActionStep):
        code_snippet = (step.code_action or "").strip()[:80]
        obs_snippet  = str(step.observations or "")[:60]
        print(f"    Code  : {code_snippet}")
        print(f"    Obs   : {obs_snippet}")
        print(f"    Tokens: {step.token_usage}")

    elif isinstance(step, FinalAnswerStep):
        print("    Output:", str(step.output)[:80])

Total steps in memory: 3

[0] TaskStep
    Task: What is 123 multiplied by 456? Use the python_interpreter tool to compute it.
[1] ActionStep
    Code  : result = 123 * 456
result
    Obs   : Execution logs:
Last output from code snippet:
56088
    Tokens: TokenUsage(input_tokens=2089, output_tokens=53, total_tokens=2142)
[2] ActionStep
    Code  : final_answer(56088)
    Obs   : Execution logs:
Last output from code snippet:
56088
    Tokens: TokenUsage(input_tokens=2205, output_tokens=46, total_tokens=2251)


The memory object is central to agent debugging. By inspecting `ActionStep.code_action` (for `CodeAgent`) or `ActionStep.tool_calls` (for `ToolCallingAgent`) you can see exactly what the model decided to do and why. `token_usage` lets you track cost per step.

## 11  Multi-step Reasoning

Some tasks cannot be solved in a single action — they require computing an intermediate value and then using it in a follow-up calculation. Here we pose such a task explicitly and watch the agent work through it over multiple steps.

In [16]:
@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit.

    Args:
        celsius: Temperature value in degrees Celsius.
    """
    return celsius * 9 / 5 + 32


@tool
def body_temp_status(fahrenheit: float) -> str:
    """Classify a body temperature given in Fahrenheit as normal, fever, or hypothermia.

    Normal body temperature is 97 – 99 °F. Above 100.4 °F is fever.
    Below 95 °F is hypothermia.

    Args:
        fahrenheit: Body temperature in degrees Fahrenheit.
    """
    if fahrenheit > 100.4:
        return "fever"
    elif fahrenheit < 95.0:
        return "hypothermia"
    else:
        return "normal"


agent_multistep = CodeAgent(
    tools=[celsius_to_fahrenheit, body_temp_status, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    max_steps=8,
)

result = agent_multistep.run(
    "A patient's temperature is 38.9 °C. "
    "Convert it to Fahrenheit, then classify the result."
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ A patient's temperature is 38.9 °C. Convert it to Fahrenheit, then classify the result.                         │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Step 1: Convert Celsius to Fahrenheit                                                                          
  celsius_temp = 38.9                                                                                              
  fahrenheit_temp = celsius_to_fahrenheit(celsius_temp)                                                            
  print(f"Temperature in Fahrenheit: {fahrenheit_temp}")                                                           
                                                                                                                   
  # Step 2: Classify the body temperature                                                                          
  temperature_status = body_temp_status(fahrenheit_temp)                                                           
  print(f"Temperature status: {temperature_status}")                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Temperature in Fahrenheit: 102.02
Temperature status: fever

Out: None

[Step 1: Duration 3.09 seconds| Input tokens: 2,099 | Output tokens: 149]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("fever")                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: fever

[Step 2: Duration 1.46 seconds| Input tokens: 4,495 | Output tokens: 199]


Final answer: fever


In [17]:
# Print a compact step-by-step trace
action_steps = [s for s in agent_multistep.memory.steps if isinstance(s, ActionStep)]
print(f"Number of action steps: {len(action_steps)}\n")

for step in action_steps:
    print(f"--- Step {step.step_number} ---")
    print("Code     :", (step.code_action or "").strip()[:120])
    print("Obs      :", str(step.observations or "")[:80])
    print()

Number of action steps: 2

--- Step 1 ---
Code     : # Step 1: Convert Celsius to Fahrenheit
celsius_temp = 38.9
fahrenheit_temp = celsius_to_fahrenheit(celsius_temp)
print(
Obs      : Execution logs:
Temperature in Fahrenheit: 102.02
Temperature status: fever
Last

--- Step 2 ---
Code     : final_answer("fever")
Obs      : Execution logs:
Last output from code snippet:
fever



The trace shows the two-step chain: the agent first called `celsius_to_fahrenheit` with 38.9, obtained the Fahrenheit value, then passed that value to `body_temp_status`. The intermediate result was preserved in the agent's context across steps — this is the key advantage over a single-turn prompt.

## 12  Advanced: Multi-Agent Systems

For complex workflows it is useful to decompose the problem across **specialised sub-agents** coordinated by an **orchestrator**. smolagents supports this via the `managed_agents` parameter: sub-agents appear as callable tools to the orchestrator, which delegates subtasks to them.

```
Orchestrator (CodeAgent)
  ├── math_agent   ← specialised for arithmetic
  └── text_agent   ← specialised for text analysis
```

Each sub-agent has a `name` and `description` that the orchestrator reads — just like tool descriptions.

In [18]:
# Sub-agent 1: arithmetic specialist
math_agent = CodeAgent(
    tools=[PythonInterpreterTool(), FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    name="math_agent",
    description=(
        "Expert at mathematical calculations. "
        "Give it a concrete arithmetic problem and it returns the numerical result."
    ),
    max_steps=4,
    verbosity_level=0,
)

# Sub-agent 2: text analysis specialist
text_agent = CodeAgent(
    tools=[text_statistics, word_frequency, FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    name="text_agent",
    description=(
        "Expert at text analysis. "
        "Give it a text passage and a question about its statistics or word frequency."
    ),
    max_steps=4,
    verbosity_level=0,
)

# Orchestrator: delegates to sub-agents
orchestrator = CodeAgent(
    tools=[FinalAnswerTool()],
    model=model,
    code_block_tags="markdown",
    managed_agents=[math_agent, text_agent],   # Sub-agents become callable tools
    max_steps=6,
)

result = orchestrator.run(
    "I have a text passage below. First, use the text agent to count its words. "
    "Then use the math agent to compute the square root of that word count. "
    "Report both numbers.\n\n"
    + corpus   # reuse the corpus variable defined in section 7
)
print("\nFinal answer:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I have a text passage below. First, use the text agent to count its words. Then use the math agent to compute   │
│ the square root of that word count. Report both numbers.                                                        │
│                                                                                                                 │
│ Natural language processing enables computers to understand human language. Language models learn statistical   │
│ patterns from large language corpora. Understanding language structure is fundamental to natural language       │
│ processing.                                                                                                     │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text_passage = "Natural language processing enables computers to understand human language. Language models      
  learn statistical patterns from large language corpora. Understanding language structure is fundamental to       
  natural language processing."                                                                                    
  word_count = text_agent(task="Please count the number of words in the following passage: " + text_passage,       
  additional_args={})                                                                                              
  print("Word Count:", word_count)                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Word Count: Here is the final answer from your managed agent 'text_agent':
{'task_outcome_short': 27, 'task_outcome_detailed': {'word_count': 27, 'sentence_count': 3, 'avg_word_length': 
7.33, 'char_count': 201}, 'additional_context': 'Understanding the word and sentence structure in a piece of text 
is critical for various applications in natural language processing, such as summarization, text analysis, and 
machine learning model training.'}

Out: None

[Step 1: Duration 10.74 seconds| Input tokens: 2,285 | Output tokens: 138]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({"word_count": 27, "square_root": square_root_of_word_count})                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'word_count': 27, 'square_root': 5.196152422706632}

[Step 4: Duration 1.80 seconds| Input tokens: 11,229 | Output tokens: 402]


Final answer: {'word_count': 27, 'square_root': 5.196152422706632}


The orchestrator treated each sub-agent as a tool, called them in the required order, and combined their outputs. This pattern scales naturally: you can add more specialised sub-agents (search, database, image analysis) without complicating the orchestrator's reasoning — it only needs to know each agent's name and description.

**Step callbacks** (another advanced feature) let you attach side-effect functions that fire after every step. They are useful for logging, early stopping, or live dashboards:

```python
def log_step(step):
    if isinstance(step, ActionStep):
        print(f"[callback] step {step.step_number} — tokens used: {step.token_usage}")

agent = CodeAgent(
    tools=[...], model=model,
    step_callbacks=[log_step],
)
```

## Summary

| Topic | Key takeaway |
|---|---|
| Agent loop | Observe -> Think -> Act, repeated until `final_answer` is called |
| `OpenAIServerModel` | Wraps the OpenAI API; set `model_id` + `api_key`, then pass it to any agent |
| `CodeAgent` | LLM writes Python; local executor runs it; very flexible |
| `ToolCallingAgent` | LLM emits JSON tool calls; cleaner for well-defined tool APIs |
| Tool descriptions | The model reads the docstring - precise descriptions = reliable tool selection |
| `@tool` decorator | Turn any annotated function into a first-class tool in one line |
| Tool composition | Pass multiple tools; the agent chains them autonomously |
| Agent memory | `agent.memory.steps` - inspect every LLM decision and observation |
| `max_steps` | Hard cap on loop iterations; catch `AgentMaxStepsError` for graceful failure |
| Multi-agent | Sub-agents with `name` + `description` become callable tools for an orchestrator |

### Suggested next steps

- **Web-enabled agents** - add `DuckDuckGoSearchTool` or `WikipediaSearchTool` and build a research assistant.
- **RAG + agents** - combine the retriever from the previous notebook with a custom `@tool` so the agent can query your knowledge base.
- **Planning** - set `planning_interval=2` on a long task and inspect `PlanningStep` objects to see how the agent revises its plan.
- **Custom executors** - try `executor_type="docker"` for isolated code execution in untrusted scenarios.
- **Evaluation** - use the smolagents benchmark helpers to measure task success rate across a dataset of tasks.

## References

* smolagents documentation - https://huggingface.co/docs/smolagents
* smolagents GitHub - https://github.com/huggingface/smolagents
* smolagents models (`OpenAIServerModel`, etc.) - https://huggingface.co/docs/smolagents/reference/models
* OpenAI models overview - https://platform.openai.com/docs/models
* ReAct: Synergising Reasoning and Acting (Yao et al., 2022) - https://arxiv.org/abs/2210.03629